# Big Data Analytics with PySpark

## Mục tiêu
Thực hành quy trình phân tích dữ liệu lớn bằng PySpark:
1. Khởi tạo Spark và đọc dữ liệu lớn.
2. Kiểm tra schema và chất lượng dữ liệu.
3. Transformation, Action, Lazy Evaluation.
4. DataFrame API và Spark SQL.
5. Join dữ liệu nhiều bảng.
6. Aggregation và Window Functions.
7. Partition, Cache và Execution Plan.
8. Xuất dữ liệu tối ưu dạng Parquet.


## 0. Cài đặt và khởi tạo Spark


In [ ]:
# Nếu dùng Colab, bỏ comment dòng sau:
# !pip install -q pyspark

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *
from pathlib import Path

spark = (SparkSession.builder
         .appName("TH03_BigDataAnalytics")
         .master("local[*]")
         .config("spark.sql.shuffle.partitions", "8")
         .getOrCreate())

print("Spark version:", spark.version)
print("Master:", spark.sparkContext.master)
print("Default parallelism:", spark.sparkContext.defaultParallelism)


### Câu hỏi
1. `SparkSession` có vai trò gì?
2. `local[*]` nghĩa là gì?
3. `spark.sql.shuffle.partitions` ảnh hưởng đến thao tác nào?


## 1. Đọc dữ liệu và Data Understanding


In [ ]:
DATA = Path("data")

events = spark.read.option("header", True).option("inferSchema", True).csv(str(DATA/"ecommerce_events_dirty.csv"))
customers = spark.read.option("header", True).option("inferSchema", True).csv(str(DATA/"customers.csv"))
products = spark.read.option("header", True).option("inferSchema", True).csv(str(DATA/"products.csv"))

print("Events:", events.count(), "rows,", len(events.columns), "columns")
print("Customers:", customers.count())
print("Products:", products.count())

events.printSchema()
events.show(5, truncate=False)


### Bài 1: Data profiling


In [ ]:
# Thực hiện:
# 1. Đếm số event theo EventType
# 2. Đếm số CustomerID và ProductID distinct
# 3. Thống kê mô tả Quantity, UnitPrice, Discount, Amount
# 4. Tìm min/max EventTime

events.groupBy("EventType").count().orderBy(F.desc("count")).show()
events.select("Quantity","UnitPrice","Discount","Amount").summary().show()


## 2. Data Quality


In [ ]:
# Missing values
missing = events.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in events.columns
])
missing.show(truncate=False)

# Duplicate theo toàn bộ dòng
total = events.count()
distinct_total = events.distinct().count()
print("Duplicate rows:", total - distinct_total)

# Giá trị bất thường
events.filter(
    (F.col("UnitPrice") <= 0) |
    (F.col("Discount") < 0) |
    (F.col("Discount") > 1) |
    (F.col("Quantity") == 0)
).show(20, truncate=False)


### Bài 2: Thiết kế quy tắc làm sạch
Giải thích trước khi viết code:
- Khi nào xóa dòng?
- Khi nào giữ giao dịch return có Quantity âm?
- Có nên điền CustomerID thiếu hay không?
- Discount > 1 có hợp lệ không?


In [ ]:
clean = (events
    .dropDuplicates(["EventID"])
    .filter(F.col("ProductID").isNotNull())
    .filter(F.col("UnitPrice").isNotNull() & (F.col("UnitPrice") > 0))
    .filter(F.col("Discount").between(0, 1))
    .filter(F.col("Quantity") != 0)
    .withColumn("EventTime", F.to_timestamp("EventTime"))
)

print("Before:", events.count())
print("After :", clean.count())


## 3. Transformation, Action và Lazy Evaluation


In [ ]:
purchase_df = (clean
    .filter(F.col("EventType") == "purchase")
    .withColumn("NetAmount",
                F.col("Quantity") * F.col("UnitPrice") * (1 - F.col("Discount")))
)

# Đây mới là logical plan; Spark chưa nhất thiết thực thi toàn bộ.
purchase_df.explain(mode="formatted")

# Action
purchase_df.select(F.sum("NetAmount").alias("Revenue")).show()


### Câu hỏi
Phân loại các thao tác sau thành Transformation/Action:
`select`, `filter`, `withColumn`, `groupBy`, `show`, `count`, `collect`, `write`.


## 4. DataFrame Analytics


In [ ]:
# KPI cơ bản
purchase_df.agg(
    F.count("*").alias("PurchaseEvents"),
    F.countDistinct("CustomerID").alias("Customers"),
    F.countDistinct("ProductID").alias("Products"),
    F.sum("Quantity").alias("Units"),
    F.sum("NetAmount").alias("Revenue"),
    F.avg("NetAmount").alias("AvgEventValue")
).show()


In [ ]:
# Phân tích theo thời gian
time_df = (purchase_df
    .withColumn("Year", F.year("EventTime"))
    .withColumn("Month", F.month("EventTime"))
    .withColumn("Date", F.to_date("EventTime"))
    .withColumn("Hour", F.hour("EventTime"))
)

time_df.groupBy("Year","Month").agg(
    F.sum("NetAmount").alias("Revenue"),
    F.count("*").alias("Purchases")
).orderBy("Year","Month").show(50)


### Bài 3: Business questions
Viết PySpark DataFrame API để trả lời:
1. Kênh nào tạo doanh thu cao nhất?
2. Khung giờ nào có nhiều purchase nhất?
3. Tháng nào doanh thu cao nhất?
4. Tỷ lệ add_to_cart → purchase (mức tổng quát) là bao nhiêu?


## 5. Join nhiều bảng


In [ ]:
enriched = (clean
    .join(customers, "CustomerID", "left")
    .join(products, "ProductID", "left")
)

enriched.select(
    "EventID","EventType","CustomerID","Region","CustomerTier",
    "ProductID","Category","Brand","Channel","Amount"
).show(10, truncate=False)


### Bài 4: Join analytics
1. Revenue theo Region.
2. Revenue theo Category.
3. Purchase theo CustomerTier.
4. Top 10 Brand theo Revenue.
5. So sánh Channel × Region.


In [ ]:
(enriched
 .filter(F.col("EventType")=="purchase")
 .groupBy("Region","Category")
 .agg(F.sum("Amount").alias("Revenue"),
      F.count("*").alias("Purchases"))
 .orderBy(F.desc("Revenue"))
 .show(30))


## 6. Spark SQL


In [ ]:
enriched.createOrReplaceTempView("events360")

spark.sql('''
SELECT
    Region,
    Category,
    SUM(CASE WHEN EventType='purchase' THEN Amount ELSE 0 END) AS Revenue,
    COUNT(CASE WHEN EventType='purchase' THEN 1 END) AS Purchases
FROM events360
GROUP BY Region, Category
ORDER BY Revenue DESC
''').show(30)


### Bài 5: Viết ít nhất 5 truy vấn Spark SQL
Bắt buộc có:
- WHERE
- GROUP BY
- ORDER BY
- JOIN hoặc dùng view đã join
- Aggregation


## 7. Window Functions


In [ ]:
purchase_enriched = (enriched
    .filter(F.col("EventType")=="purchase")
    .withColumn("Revenue", F.col("Quantity")*F.col("UnitPrice")*(1-F.col("Discount")))
)

w_customer = Window.partitionBy("CustomerID").orderBy("EventTime")

customer_timeline = (purchase_enriched
    .withColumn("PrevPurchaseTime", F.lag("EventTime").over(w_customer))
    .withColumn("DaysSincePrevPurchase",
                F.datediff("EventTime", "PrevPurchaseTime"))
)

customer_timeline.select(
    "CustomerID","EventTime","PrevPurchaseTime","DaysSincePrevPurchase","Revenue"
).show(20, truncate=False)


### Bài 6: Window challenge


In [ ]:
# 1. Xếp hạng sản phẩm theo Revenue trong từng Category.
# 2. Tính cumulative revenue theo thời gian.
# 3. Tìm lần mua đầu tiên và gần nhất của mỗi khách hàng.
# 4. Tính khoảng cách ngày trung bình giữa các lần mua.


## 8. Partition, Cache và Performance


In [ ]:
print("Partitions before:", purchase_enriched.rdd.getNumPartitions())

repart = purchase_enriched.repartition(8, "Category")
print("After repartition:", repart.rdd.getNumPartitions())

repart.cache()
print("Cached count:", repart.count())

repart.groupBy("Category").agg(F.sum("Revenue").alias("Revenue")).explain(mode="formatted")


### Câu hỏi hiệu năng
1. `repartition()` khác `coalesce()` thế nào?
2. Khi nào `cache()` hữu ích?
3. Shuffle là gì?
4. Vì sao `collect()` nguy hiểm với dữ liệu lớn?
5. Vì sao Parquet thường phù hợp hơn CSV cho analytics?


## 9. Xuất Parquet


In [ ]:
OUT = Path("output")
OUT.mkdir(exist_ok=True)

(enriched
 .write
 .mode("overwrite")
 .partitionBy("EventType")
 .parquet(str(OUT/"events360_parquet")))

print("Saved to:", OUT/"events360_parquet")


## 10. Mini Analytics Pipeline


In [ ]:
# Tạo bảng summary phục vụ BI/TH04
monthly_summary = (purchase_enriched
    .withColumn("YearMonth", F.date_format("EventTime","yyyy-MM"))
    .groupBy("YearMonth","Region","Category","Channel")
    .agg(
        F.sum("Revenue").alias("Revenue"),
        F.sum("Quantity").alias("Units"),
        F.count("*").alias("PurchaseEvents"),
        F.countDistinct("CustomerID").alias("Customers")
    )
    .withColumn("RevenuePerCustomer",
                F.col("Revenue") / F.when(F.col("Customers")>0, F.col("Customers")))
)

monthly_summary.orderBy("YearMonth","Region","Category").show(30, truncate=False)


## 11. Bài tập tổng hợp
Thực hiện và trình bày:

1. **Data Quality Report**: ít nhất 5 vấn đề dữ liệu.
2. **5 KPI chính**.
3. **5 Spark SQL queries**.
4. **3 phân tích dùng Join**.
5. **2 phân tích dùng Window Function**.
6. So sánh thời gian xử lý trước/sau `cache()` cho một phép tính lặp.
7. Xuất kết quả dưới dạng Parquet.
8. Viết **5 Business Insights** dựa trên kết quả định lượng.

### Câu hỏi thảo luận
- DataFrame khác Pandas DataFrame ở điểm nào quan trọng nhất?
- Vì sao Spark phù hợp dữ liệu lớn hơn Pandas?
- Spark có luôn nhanh hơn Pandas không?
- Khi nào không nên dùng Spark?


## 12. Yêu cầu nộp
- `BigDataPySpark_HoTen_MSSV.ipynb`
- ảnh/chụp kết quả chạy hoặc HTML/PDF notebook (nếu yêu cầu)
- thư mục `output/` hoặc ảnh minh họa kết quả Parquet
- phần kết luận gồm tối thiểu 5 insight và 3 đề xuất kinh doanh.

**Không dùng `collect()` để lấy toàn bộ dữ liệu về driver.**
